In [ ]:
!pip -q install matplotlib
!pip -q install scipy
!pip -q install mne
!pip -q install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 101.8 MB/s eta 0:00:00


In [ ]:
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
import pywt
import mne
import numpy as np


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = '/content/drive/MyDrive/Files_CortexCodec/S001R01.edf'

In [ ]:
raw = mne.io.read_raw(file_path, preload=True)
raw.plot(n_channels = 25, scalings='auto', title='Raw EEG Data Visualization')

In [ ]:
info = raw.info
print(info)

In [ ]:
fs = 160

In [ ]:
raw.filter(l_freq=1, h_freq=40)

In [ ]:
raw.plot(n_channels = 5, scalings='auto', title='Filtered EEG Data Visualization')

In [ ]:
print(raw.ch_names)

In [ ]:
channel_name = raw.ch_names[0]
print(channel_name)

In [ ]:
from mne.preprocessing import ICA

ica = mne.preprocessing.ICA(n_components=15, random_state=97, max_iter=800)

ica.fit(raw)

In [ ]:
ica.plot_sources(raw)

In [ ]:
ica.exclude = [0, 7]

In [ ]:
raw_clean = ica.apply(raw.copy())

In [ ]:
data_raw = raw.get_data()[0]
data_clean = raw_clean.get_data()[0]

In [ ]:
time = np.arange(len(data_raw)) / fs

In [ ]:
print("Analyzing channel: " +  "Fc5.")
print("Sampling rate: " + " 160.0" + "Hz")

In [ ]:
raw = mne.io.read_raw_edf(file_path, preload=True)
data_raw = raw.get_data()[0]

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(time, data_raw * 1e6, label='Raw', color='red')
plt.plot(time, data_clean * 1e6, label='Cleaned', color='blue')

plt.title('EEG Comparison – Channel: ' + str("Fc5."))
plt.xlabel('Time (s)')
plt.ylabel('Amplitude (µV)')
plt.legend()


plt.show()

In [ ]:
wavelet = 'db4'
decomp_level = 2
coeffs = pywt.wavedec(data_clean, wavelet, decomp_level)


In [ ]:
arr_len = max(len(c) for c in coeffs)
pad_coeffs = [np.pad(c, (0, arr_len - len(c))) for c in coeffs]
coeff_matrix = np.array(pad_coeffs)

In [ ]:
plt.imshow(np.abs(coeff_matrix), aspect='auto',
           extent=[0, len(data_clean)/fs, len(coeffs), 0])
plt.colorbar(label='|Coefficient Magnitude|')
plt.title(f"DWT Energy Map - {channel_name} (ICA Cleaned)")
plt.xlabel("Time (s)")
plt.ylabel("Decomposition Level")
plt.tight_layout()
plt.show()


In [ ]:
f, t, psd = spectrogram(data_clean, 160, nperseg=256, noverlap=128)


In [ ]:

plt.figure(figsize=(15, 4))
plt.pcolormesh(t, f, 10 * np.log10(psd + 1e-10), shading='gouraud', cmap='jet')
plt.title(f"Spectrogram (STFT) - {channel_name}")
plt.ylabel('Frequency (Hz)')
plt.xlabel('Time (s)')
plt.ylim(0, 5)  # focus on EEG
plt.colorbar(label='Power (dB)')

plt.show()